## In this notebook, we test the model (which was trained on _Corpus Nummorum_ data) on three _Numismatics_ datasets (Seleucid, BIGR, and PELLA)

Plan of attack
* Downloading fine-tuned ImageNet 21k model checkpoint from .pth file
* Saving image and motif metadata for three additional datasets
* Checking the contents of the saved files
* Testing the model on the new datasets
* Recording average precision (AP) for each motif and mean average precision (mAP) for each of the three datasets

Import statements and necessary functions

In [ ]:
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import timm
from timm.data import resolve_model_data_config, create_transform, resolve_data_config
from timm.data.transforms_factory import create_transform
from tqdm.auto import tqdm
from tqdm import tqdm
import copy
import numpy as np
from sklearn.metrics import average_precision_score
import tarfile
import os
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

In [ ]:
# from google.colab import runtime
# runtime.unassign()
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class CoinMotifDataset(Dataset):
    def __init__(self, frame, motif_cols, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.motif_cols = motif_cols
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]

        img = Image.open(row["image_path"]).convert("RGB")

        if self.transform:
            img = self.transform(img)

        y = torch.tensor(row[self.motif_cols].values.astype("float32"))

        return img, y

In [ ]:
# device = "cuda" if torch.cuda.is_available() else "cpu"
# model = model.to(device)

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")

def macro_ap(y_true, y_prob, motif_cols):
    scores = []
    for j, motif in enumerate(motif_cols):
        if y_true[:, j].sum() > 0:
            scores.append(average_precision_score(y_true[:, j], y_prob[:, j]))
    return float(np.mean(scores)) if scores else np.nan

def train_one_epoch(model, loader, optimizer, criterion, device, epoch, phase):
    model.train()
    total_loss = 0.0

    pbar = tqdm(loader, desc=f"{phase} epoch {epoch}", leave=False)

    for imgs, y in pbar:
        imgs = imgs.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(imgs)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader.dataset)

@torch.no_grad()
def predict(model, loader, device, desc="Validate"):
    model.eval()

    probs_all = []
    y_all = []

    pbar = tqdm(loader, desc=desc, leave=False)

    for imgs, y in pbar:
        imgs = imgs.to(device, non_blocking=True)

        logits = model(imgs)
        probs = torch.sigmoid(logits).cpu()

        probs_all.append(probs)
        y_all.append(y.cpu())

    return torch.cat(probs_all).numpy(), torch.cat(y_all).numpy()

def run_training_phase(model, train_loader, val_loader, optimizer, criterion, device,
                       num_epochs, phase, best_score=-np.inf, best_state=None):
    for epoch in tqdm(range(1, num_epochs + 1), desc=phase):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            epoch=epoch,
            phase=phase,
        )

        val_probs, val_y = predict(
            model=model,
            loader=val_loader,
            device=device,
            desc=f"{phase} validation {epoch}",
        )

        val_ap = macro_ap(val_y, val_probs, sample_motifs)

        print(f"{phase} epoch {epoch}: train_loss={train_loss:.4f}, val_macro_AP={val_ap:.4f}")

        if val_ap > best_score:
            best_score = val_ap
            best_state = copy.deepcopy(model.state_dict())
            print(f"  New best val_macro_AP: {best_score:.4f}")

    return best_score, best_state

Investigating .tar file of Corpus Nummorum images

In [ ]:
tar_path = "/content/drive/MyDrive/coins_project/coin_images.tar"

with tarfile.open(tar_path, "r:*") as tar:
    tar.list()

"Scraping" databases for coin images and metadata

In [ ]:
# BASE_URL = "https://numismatics.org/sco/results?q=&start={}"
# BASE_URL = "https://numismatics.org/pella/results?q=&start={}"
BASE_URL = "https://numismatics.org/bigr/results?q=&start={}"

OUTPUT_DIR = OUTPUT_DIR = "/content/drive/MyDrive/coins_project"
# IMAGE_DIR = os.path.join(OUTPUT_DIR, "seleucid_images")
# IMAGE_DIR = os.path.join(OUTPUT_DIR, "pella_images")
IMAGE_DIR = os.path.join(OUTPUT_DIR, "bigr_images")

os.makedirs(IMAGE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


def parse_page(start):

    url = BASE_URL.format(start)

    r = session.get(url, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "html.parser")

    page_records = []

    for coin in soup.select("div.result-doc"):

        # records ID

        h4 = coin.find("h4")
        if h4 is None:
            continue

        a = h4.find("a")
        if a is None:
            continue

        record_id = a["href"].split("/")[-1]

        # gets image

        image_url = None

        thumb = coin.select_one("a.thumbImage")

        if thumb is not None:
            image_url = thumb.get("href")

        # gets metadata

        metadata = {}

        for dt, dd in zip(
            coin.find_all("dt"),
            coin.find_all("dd")
        ):
            metadata[
                dt.get_text(strip=True)
            ] = dd.get_text(" ", strip=True)

        page_records.append({

            "RecordId": record_id,

            "ImageURL": image_url,

            "Obverse": metadata.get("Obverse"),

            "Reverse": metadata.get("Reverse"),

            "Date": metadata.get("Date"),

            "Denomination": metadata.get("Denomination"),

            "Weight": metadata.get("Weight (in g)")
        })

    return page_records


# downloads image

def download_image(url, filename):

    if url is None:
        return False

    try:

        r = session.get(url, timeout=30)

        if r.status_code != 200:
            return False

        with open(filename, "wb") as f:
            f.write(r.content)

        return True

    except Exception:

        return False



# crawls site

records = []

TOTAL = 5000
PAGE_SIZE = 20

for start in tqdm(range(0, TOTAL, PAGE_SIZE)):

    page = parse_page(start)

    for coin in page:

        # downloads image

        image_name = f'{coin["RecordId"]}.jpg'

        image_path = os.path.join(
            IMAGE_DIR,
            image_name
        )

        success = download_image(
            coin["ImageURL"],
            image_path
        )

        if not success:
            image_name = None

        # stores metadata

        records.append({

            "RecordId": coin["RecordId"],

            "Image": image_name,

            "Obverse": coin["Obverse"],

            "Reverse": coin["Reverse"],

            "Date": coin["Date"],

            "Denomination": coin["Denomination"],

            "Weight": coin["Weight"]

        })

    # delay for server
    time.sleep(0.1)

# saves dataset

df = pd.DataFrame(records)

df.to_csv(
    # os.path.join(OUTPUT_DIR, "seleucid_coins.csv"),
    # os.path.join(OUTPUT_DIR, "pella_coins.csv"),
    os.path.join(OUTPUT_DIR, "bigr_coins.csv"),
    index=False
)

print(df.head())
print()
print("Total records:", len(df))

Investigates the .tar paths of other coin datasets

In [ ]:
# src = Path("/content/drive/MyDrive/coins_project/seleucid_images")  # your Drive folder containing motif/image folders
# tar_path = Path("/content/drive/MyDrive/coins_project/seleucid_images.tar")
# src = Path("/content/drive/MyDrive/coins_project/pella_images")  # your Drive folder containing motif/image folders
# tar_path = Path("/content/drive/MyDrive/coins_project/pella_images.tar")
src = Path("/content/drive/MyDrive/coins_project/bigr_images")  # your Drive folder containing motif/image folders
tar_path = Path("/content/drive/MyDrive/coins_project/bigr_images.tar")

print("source:", src)
print("tar:", tar_path)

!tar -czf "$tar_path" -C "$src" .

Moving on to booting up the deep learning model using a .pth file and testing it on new datasets

In [ ]:
checkpoint_path = "/content/drive/MyDrive/coins_project/coin_motif_vit_checkpoint.pth"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
num_motifs = 7

model = timm.create_model(
    "vit_base_patch16_224.augreg_in21k",
    pretrained=False,
    num_classes=num_motifs,
)

model.to(device)

In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)

In [ ]:
print(checkpoint.keys())

In [ ]:
model.load_state_dict(checkpoint["model_state_dict"])

In [ ]:
# evaluates model
model.eval()

In [ ]:
config = resolve_data_config({}, model=model)

transform = create_transform(
    **config,
    is_training=False,
)

Test testing procedure on a random image

In [ ]:
# image = Image.open("/content/drive/MyDrive/coins_project/CoinsDataset/CN_dataset_nlp_objects/bull/CN_type_1417_cn_coin_2220_p_rev.jpg").convert("RGB")
image = Image.open("/content/drive/MyDrive/coins_project/pella_images/pella.philip_ii.10.jpg").convert("RGB")

x = transform(image).unsqueeze(0).to(device)

In [ ]:
with torch.no_grad():
    logits = model(x)

probabilities = torch.softmax(logits, dim=1)

confidence, prediction = probabilities.max(dim=1)

print(prediction.item())
print(confidence.item())

Looks good!

Lastly, test the model on the other coin datasets

In [ ]:
base_path = Path('/content/drive/MyDrive/coins_project')

pella_copy_path = base_path / 'pella_coins_copy.csv'
pella_csv_path = base_path / 'pella_coins.csv'
bigr_csv_path = base_path / 'bigr_coins.csv'
seleucid_csv_path = base_path / 'seleucid_coins.csv'
pella_images_path = base_path / 'pella_images'
bigr_images_path = base_path / 'bigr_images'
seleucid_images_path = base_path / 'seleucid_images'
cn_nlp_path = base_path / 'CoinsDataset' / 'CN_dataset_nlp_objects'

if base_path.is_dir():
  print(f"Found directory: {base_path}")
else:
  print(f"Directory not found at {base_path}")
for p in [pella_copy_path, pella_csv_path, bigr_csv_path]:
  if p.is_file():
    print(f"Found file: {p}")
  else:
    print(f"File not found at {p}")
for p in [pella_images_path, bigr_images_path, seleucid_images_path, cn_nlp_path]:
  if p.is_dir():
    print(f"Found directory: {p}")
  else:
    print(f"Directory not found at {p}")

Found directory: /content/drive/MyDrive/coins_project
Found file: /content/drive/MyDrive/coins_project/pella_coins_copy.csv
Found file: /content/drive/MyDrive/coins_project/pella_coins.csv
Found file: /content/drive/MyDrive/coins_project/bigr_coins.csv
Found directory: /content/drive/MyDrive/coins_project/pella_images
Found directory: /content/drive/MyDrive/coins_project/bigr_images
Found directory: /content/drive/MyDrive/coins_project/seleucid_images
Found directory: /content/drive/MyDrive/coins_project/CoinsDataset/CN_dataset_nlp_objects


In [ ]:
# collects all the motifs in the CN dataset and how often they appear.
# takes a few minutes to run
motifs_with_counts = {}
for motif in os.listdir(cn_nlp_path):
  p = cn_nlp_path / motif
  if p.is_dir():
    motifs_with_counts[motif] = len(os.listdir(cn_nlp_path / motif))

print(motifs_with_counts)

{'abacus': 5, 'abundantia': 3, 'acrostolium': 11, 'acroteria': 1, 'actaeon': 3, 'aegis': 432, 'aeneas': 57, 'aequitas': 180, 'agonistic_crown': 2, 'agrippa': 7, 'agrippina_minor': 2, 'alexander_iii': 145, 'altar': 1049, 'amphora': 247, 'anchialos': 2, 'anchises': 60, 'anchor': 32, 'androclus': 3, 'andromeda': 3, 'animal': 13, 'annona': 2, 'antinous': 5, 'antiochus_ii_theos': 2, 'antlers': 24, 'antonia_minor': 6, 'antoninus_pius': 892, 'anubis': 15, 'aphrodite': 65, 'apis': 35, 'aplustre': 84, 'apollo': 2774, 'apollon': 9, 'apple': 201, 'arch': 12, 'archer': 2, 'ares': 224, 'ariadne': 7, 'arm': 1907, 'armour': 38, 'arrow': 490, 'artemis': 923, 'ascanius': 57, 'asclepius': 1029, 'astragal': 6, 'athena': 4108, 'athlete': 115, 'attis': 7, 'augustus': 384, 'aulos': 1, 'bag': 3, 'barley': 49, 'base': 268, 'basin': 14, 'basket': 90, 'bear': 19, 'bee': 29, 'beehive': 6, 'belt': 4, 'berry': 82, 'biga': 125, 'bird': 49, 'boar': 144, 'board': 65, 'bonus_eventus': 102, 'boot': 138, 'bow': 1509, 'b

In [ ]:
# Reduce to the most frequently appearing motifs.
# This seems to be about the cutoff in the model-training/testing file.
# That is, it's about 0.1% of the length of the CN dataset.
MIN_MOTIF_FREQUENCY = 150
frequent_motifs = {motif: count for (motif,count) in motifs_with_counts.items() if count >= MIN_MOTIF_FREQUENCY}
print(frequent_motifs)

{'aegis': 432, 'aequitas': 180, 'altar': 1049, 'amphora': 247, 'antoninus_pius': 892, 'apollo': 2774, 'apple': 201, 'ares': 224, 'arm': 1907, 'arrow': 490, 'artemis': 923, 'asclepius': 1029, 'athena': 4108, 'augustus': 384, 'base': 268, 'bow': 1509, 'branch': 939, 'bull': 1342, 'bust': 9423, 'caduceus': 596, 'cantharus': 617, 'cap': 191, 'caracalla': 2238, 'chlamys': 352, 'cista': 507, 'club': 1014, 'column': 255, 'commodus': 947, 'corn': 1027, 'cornucopia': 1703, 'corn_wreath': 530, 'crepidoma': 373, 'crescent': 261, 'crispina': 159, 'cuirass': 5257, 'cybele': 318, 'demeter': 971, 'diadem': 1455, 'diadumenian': 161, 'dionysus': 1455, 'dolphin': 1198, 'domitian': 209, 'double_ax': 154, 'double_chiton': 182, 'eagle': 1564, 'ear': 694, 'earring': 632, 'elagabalus': 798, 'emperor': 636, 'faustina_minor': 391, 'figure': 172, 'foot': 1901, 'gallienus': 154, 'garment': 292, 'geta': 682, 'gordian': 735, 'gorgoneion': 450, 'grain': 485, 'grape': 1165, 'griffin': 1152, 'hadrian': 296, 'hair': 7

In [ ]:
# Functions to import the numismatics.org datasets and augment the CSV files
# with dedicated columns for motifs.

def open_numismatics_data(csv_path: Path, images_path: Path) -> pd.DataFrame:
  df = pd.read_csv(csv_path)
  df.rename(columns={"Image": "filename"}, inplace=True)
  # drops weights column, which is all na.
  df.dropna(axis='columns', how='all', inplace=True)
  # drops any rows with no image.
  df.dropna(axis='rows', subset="filename", inplace=True)

  df["image_path"] = images_path
  df["image_path"] = df["image_path"] / df["filename"]

  return df

def check_for_motifs(df: pd.DataFrame, motifs: list[str], column_to_check: str = "Reverse") -> pd.DataFrame:
  if column_to_check in df.columns:
    # adds all columns at once instead of one by one to avoid errors
    motif_columns = pd.DataFrame(columns=motifs, dtype=str)
    result = pd.concat([df, motif_columns], axis='columns')
    for motif in motifs:
      result[motif] = result[column_to_check].str.contains("(?i)" + motif).astype(int)
    return result
  else:
    raise KeyError(f"DataFrame {df} has no column {column_to_check}.")

In [ ]:
# Let's see what we're working with...

df = open_numismatics_data(bigr_csv_path, bigr_images_path)
df.head(10)

,RecordId,filename,Obverse,Reverse,Date,Denomination,image_path
0,bigr.diodotus_i_ii.1,bigr.diodotus_i_ii.1.jpg,"Head of Diodotus right, diademed",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Zeus ...,mid third century BCE,Stater,/content/drive/MyDrive/coins_project/bigr_imag...
1,bigr.diodotus_i_ii.2,bigr.diodotus_i_ii.2.jpg,"Head of Diodotus right, diademed",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Zeus ...,mid third century BCE,Tetradrachm,/content/drive/MyDrive/coins_project/bigr_imag...
2,bigr.diodotus_i_ii.3,bigr.diodotus_i_ii.3.jpg,"Head of Diodotus right, diademed",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Zeus ...,mid third century BCE,Drachma,/content/drive/MyDrive/coins_project/bigr_imag...
5,bigr.diodotus_i_ii.5,bigr.diodotus_i_ii.5.jpg,"Head of Hermes right, wearing petasos",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Athen...,mid third century BCE,Denomination B (double),/content/drive/MyDrive/coins_project/bigr_imag...
6,bigr.diodotus_i_ii.6,bigr.diodotus_i_ii.6.jpg,"Head of Hermes right, wearing petasos",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Athen...,mid third century BCE,Denomination C (unit),/content/drive/MyDrive/coins_project/bigr_imag...
7,bigr.diodotus_i_ii.7,bigr.diodotus_i_ii.7.jpg,"Head of Hermes right, wearing petasos",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Athen...,mid third century BCE,Denomination D (half),/content/drive/MyDrive/coins_project/bigr_imag...
8,bigr.diodotus_i_ii.8,bigr.diodotus_i_ii.8.jpg,"Head of Zeus right, laureate",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Artem...,mid third century BCE,Denomination B (double),/content/drive/MyDrive/coins_project/bigr_imag...
10,bigr.diodotus_i_ii.9,bigr.diodotus_i_ii.9.jpg,"Head of Zeus right, laureate",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Artem...,mid third century BCE,Denomination B (double),/content/drive/MyDrive/coins_project/bigr_imag...
11,bigr.diodotus_i_ii.10,bigr.diodotus_i_ii.10.jpg,"Head of Zeus right, laureate",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Artem...,mid third century BCE,Denomination C (unit),/content/drive/MyDrive/coins_project/bigr_imag...
14,bigr.diodotus_i_ii.13,bigr.diodotus_i_ii.13.jpg,Eagle facing right,ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Quiver,mid third century BCE,Denomination E (quarter),/content/drive/MyDrive/coins_project/bigr_imag...


In [ ]:
df = check_for_motifs(df, frequent_motifs.keys(), "Obverse")
df.head(20)

,RecordId,filename,Obverse,Reverse,Date,Denomination,image_path,aegis,aequitas,altar,...,tunny,tyche,urn,veil,vine,wheel,wing,woman,wreath,zeus
0,bigr.diodotus_i_ii.1,bigr.diodotus_i_ii.1.jpg,"Head of Diodotus right, diademed",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Zeus ...,mid third century BCE,Stater,/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,bigr.diodotus_i_ii.2,bigr.diodotus_i_ii.2.jpg,"Head of Diodotus right, diademed",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Zeus ...,mid third century BCE,Tetradrachm,/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bigr.diodotus_i_ii.3,bigr.diodotus_i_ii.3.jpg,"Head of Diodotus right, diademed",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Zeus ...,mid third century BCE,Drachma,/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,bigr.diodotus_i_ii.5,bigr.diodotus_i_ii.5.jpg,"Head of Hermes right, wearing petasos",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Athen...,mid third century BCE,Denomination B (double),/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,bigr.diodotus_i_ii.6,bigr.diodotus_i_ii.6.jpg,"Head of Hermes right, wearing petasos",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Athen...,mid third century BCE,Denomination C (unit),/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,bigr.diodotus_i_ii.7,bigr.diodotus_i_ii.7.jpg,"Head of Hermes right, wearing petasos",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Athen...,mid third century BCE,Denomination D (half),/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,bigr.diodotus_i_ii.8,bigr.diodotus_i_ii.8.jpg,"Head of Zeus right, laureate",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Artem...,mid third century BCE,Denomination B (double),/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,1
10,bigr.diodotus_i_ii.9,bigr.diodotus_i_ii.9.jpg,"Head of Zeus right, laureate",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Artem...,mid third century BCE,Denomination B (double),/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,1
11,bigr.diodotus_i_ii.10,bigr.diodotus_i_ii.10.jpg,"Head of Zeus right, laureate",ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Artem...,mid third century BCE,Denomination C (unit),/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,1
14,bigr.diodotus_i_ii.13,bigr.diodotus_i_ii.13.jpg,Eagle facing right,ΒΑΣΙΛΕΩΣ ΔΙΟΔΟΤΟΥ\n : Quiver,mid third century BCE,Denomination E (quarter),/content/drive/MyDrive/coins_project/bigr_imag...,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
motif_counts = df[frequent_motifs.keys()].sum()
motif_counts[motif_counts >= 5]

,0
aegis,20
apollo,27
arm,26
arrow,20
athena,11
bow,21
bull,41
bust,196
club,5
cuirass,20


In [ ]:
1/0

In [ ]:
test_df = df.copy()

sample_motifs = ["eagle", "throne", "snake", "bull", "horse", "star", "head"]

data_config = resolve_model_data_config(model)
eval_tfms = create_transform(**data_config, is_training=False)

test_ds = CoinMotifDataset(test_df, sample_motifs, eval_tfms)

batch_size = 32

test_loader = DataLoader(
    test_ds,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

In [ ]:
test_probs, test_y = predict(
    model=model,
    loader=test_loader,
    device=device,
    desc="Test",
)

for j, motif in enumerate(sample_motifs):
    positives = int(test_y[:, j].sum())

    if positives > 0:
        ap = average_precision_score(test_y[:, j], test_probs[:, j])
        print(f"{motif}: AP={ap:.4f}, positives={positives}")
    else:
        print(f"{motif}: AP=undefined, positives=0")

Results of testing model on Corpus Nummorum Greco-Roman coins as well as Numismatics PELLA/BIGR/Seleucid coins

In [ ]:
# # CORPUS NUMMORUM (mAP = 0.824)
# eagle: AP=0.7564, positives=317
# throne: AP=0.8305, positives=281
# snake: AP=0.7304, positives=479
# bull: AP=0.8280, positives=265
# horse: AP=0.8858, positives=493
# star: AP=0.8288, positives=122
# head: AP=0.9110, positives=3612

In [ ]:
# # BIGR (mAP = 0.384)
# eagle: AP=0.0588, positives=1
# throne: AP=0.4306, positives=3
# snake: AP=undefined, positives=0
# bull: AP=0.1115, positives=49
# horse: AP=1.0000, positives=3
# star: AP=0.1102, positives=2
# head: AP=0.5916, positives=75

In [ ]:
# # PELLA (mAP = 0.505)
# eagle: AP=undefined, positives=0
# throne: AP=undefined, positives=0
# snake: AP=0.0103, positives=7
# bull: AP=undefined, positives=0
# horse: AP=undefined, positives=0
# star: AP=undefined, positives=0
# head: AP=0.9999, positives=3289

In [ ]:
# # SELEUCID (mAP = 0.386)
# eagle: AP=undefined, positives=0
# throne: AP=0.8209, positives=13
# snake: AP=undefined, positives=0
# bull: AP=0.0349, positives=9
# horse: AP=0.1120, positives=10
# star: AP=0.0107, positives=10
# head: AP=0.9533, positives=1062